# Урок 13 - Памет на агента


## Настройка

Този бележник демонстрира как да се създаде агент за резервации на пътувания с **постоянна памет** чрез **Microsoft Agent Framework** (MAF).

Ще научите как различните типове памет на агента — работна, краткосрочна и дългосрочна — влияят върху това как агентът съхранява и използва информацията в рамките на разговорите.

**Изисквания:**
- Проект в Microsoft Foundry с внедрен чат модел (напр. `gpt-5-mini`).
- Вход в Azure CLI — изпълнете `az login` в терминала си.
- `AZURE_AI_PROJECT_ENDPOINT` — крайна точка на вашия проект в Microsoft Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — името на вашия внедрен модел.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Видове памет на агент

AI агентите могат да използват различни видове памет, като всеки служи за различна цел:

### Работна памет
Самата нишка на разговора — съобщенията, обменени в една сесия. Агентът може да се позовава на по-ранни съобщения в същата нишка, за да поддържа последователност. В MAF това се създава чрез **`agent.create_session()`**, което връща `AgentSession`.

### Краткосрочна памет
Информация, която се запазва за продължителността на задача или сесия, но не се съхранява постоянно. Например, агентът може да натрупва факти по време на многократен планиращ разговор и да ги използва за създаване на финален маршрут.

### Дългосрочна памет
Предпочитания и факти, които се запазват **през няколко сесии**. Връщащ се потребител не трябва да повтаря своите диетични ограничения или стил на пътуване. Дългосрочната памет обикновено се съхранява във външен носител — база данни, файл или векторен индекс — и се предоставя на агента чрез инструменти.


## Работна памет със сесии

Най-простата форма на памет е сесията на разговора. Когато подадете една и съща сесия (създадена чрез `agent.create_session()`) в поредните извиквания на `agent.run()`, агентът вижда цялата история на този разговор и може да си спомни по-ранни подробности.

Нека създадем туристически агент и да демонстрираме работна памет.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Агентът правилно си спомни бюджета, защото и двете съобщения споделят една и съща сесия. Това е **работна памет** — тя съществува само през времето на сесията.

### Какво се случва с нов разговор?

Ако създадем **нова** сесия, агентът няма спомен за предишния разговор:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Модел за дългосрочна памет

За да запомним потребителските предпочитания **през сесиите**, ни е необходимо постоянно хранилище, което да живее извън нишката на разговора. Актьорът (агентът) осъществява достъп до това хранилище чрез **инструменти** — функции, които може да извиква за запазване и извличане на информация.

По-долу имплементираме прост магазин за предпочитания в паметта (в продукция бихте използвали база данни или векторен индекс) и го излагаме като инструменти, които агентът може да използва.

### Архитектура
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Сценарий 1 — Потребител за първи път резервира пътуване за годишнина

Сара посещава за първи път. Агентът трябва да запази нейните предпочитания чрез инструментите и да ги използва, за да препоръча хотели.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Сценарий 2 — Сара се връща седмици по-късно

Сара започва **съвсем нова нишка** (симулирайки нова сесия). Работната памет е празна, но хранилището за дългосрочни предпочитания все още съдържа нейната информация. Агентът трябва да я извлече и използва, за да персонализира препоръките.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резюме

В този урок научихте три типа памет на агент и как да ги имплементирате с Microsoft Agent Framework:

| Тип памет | Механизъм в MAF | Времетраене |
|---|---|---|
| **Работна** | `agent.create_session()` | Единичен разговор |
| **Късосрочна** | Натрупан контекст в рамките на нишка | Една задача / сесия |
| **Дългосрочна** | Външно хранилище, достъпвано чрез функции `@tool` | Между сесии |

### Основни изводи
1. **`agent.create_session()`** предоставя работна памет — агентът вижда цялата история на разговора в рамките на сесията.
2. **Новите сесии губят контекст** — без дългосрочна памет агентът не може да си спомня минали разговори.
3. **Функциите `@tool`** запълват пропуските — те позволяват на агента да съхранява и извлича информация от постоянно хранилище.
4. **Персонализацията се подобрява с времето** — колкото повече предпочитания се съхраняват, толкова по-добри са препоръките на агента.

### Приложения в реалния свят
- **Обслужване на клиенти**: Запомняне на историята и предпочитанията на клиента
- **Лични асистенти**: Поддържане на контекст през дни или седмици
- **Здравеопазване**: Следене на информация и предпочитания на пациентите
- **Електронна търговия**: Персонализирано пазаруване въз основа на историята

### Следващи стъпки
- Замяна на речника в паметта с база данни или векторно хранилище (например Azure AI Search)
- Добавяне на изтичане на паметта за времевочувствителна информация
- Изграждане на мултиагентни системи с обща памет
- Проучване на тетрадката Cognee за памет, базирана на граф на знания


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
